# Inspecting the Vietnam TB run

This notebook runs the summer4 port of the `tweak_cases_calib` Vietnam model at the default parameters. It is a forward run only. There is no calibration and no likelihood.

The model has 80 compartments: seven disease states and eight ages (`0, 3, 5, 10, 15, 18, 40, 65`), with infectiousness and clinical status only on `active`. Solver time is an index, `0` through `300`. Calendar year is that index plus 1800, so the saved series runs from 1800 to 2100. The population starts at 1000 people in `mtb_naive` at each age. Later births come from the UN increment series, not from that flat start.

In VS Code, select the kernel **tb-macro-summer4 (pixi)**. That kernel is the default pixi environment (`.pixi/envs/default`), which installs summer4. If it is missing, run `pixi run install-kernel` once from the repository root. Do not use the `reference` environment here: it pins summer3 and does not install this package.

`run(params)` is the corrected docstring model. The summer3 goldens still include three bugs; parity tests pass `match_golden=True`. [The next notebook](02-ReleaseModel.ipynb) plots both side by side. The bugs are written up in `reference/summer3-bugs.md`.


## Run the default parameters

`BASE_PARAMS` is the inspect-notebook default. Active case finding coverage is 0, so the `acf` flow is identically zero. Infection is seeded by a triangular pulse centred on 1830. Passive detection stays at zero until 1957, then follows the cosine scale-up through 1986, 2007 and 2014, with a dip in 2021.


In [ ]:
import numpy as np
import plotly.graph_objects as go

from tb_macro.model import calendar_years, run
from tb_macro.outputs import age_incidence, disease_state_totals
from tb_macro.parameters import BASE_PARAMS

result = run(BASE_PARAMS)
years = np.asarray(calendar_years())
totals = {state: np.asarray(values) for state, values in disease_state_totals(result).items()}


## Disease-state totals

Each line sums one disease state over age. For `active`, that sum also folds in infectiousness and clinical status. The seed in the 1830s is what first moves people out of `mtb_naive`. Treatment stays empty until detection begins in 1957, then tracks clinical disease through the scale-up.


In [ ]:
figure = go.Figure()
for state, values in totals.items():
    figure.add_trace(go.Scatter(x=years, y=values, name=state, mode="lines"))
figure.update_layout(
    title="Disease-state totals at the default parameters",
    xaxis_title="Year",
    yaxis_title="People",
    template="plotly_white",
    legend_title="Disease state",
    height=520,
)
figure.show()


## Incidence and prevalence

Prevalence here is `active` plus `treatment`. Incidence is the progression flow, summed over age. This is the corrected default, so these series will not sit on the summer3 golden parquet. For that comparison, see [the next notebook](02-ReleaseModel.ipynb) or `tests/test_parity.py` (`match_golden=True`).


In [ ]:
prevalence = totals["active"] + totals["treatment"]
incidence = sum(np.asarray(values) for values in age_incidence(result).values())

figure = go.Figure()
figure.add_trace(go.Scatter(x=years, y=prevalence, name="Prevalence", mode="lines"))
figure.add_trace(go.Scatter(x=years, y=incidence, name="Incidence", mode="lines"))
figure.update_layout(
    title="Prevalence and incidence at the default parameters",
    xaxis_title="Year",
    yaxis_title="People, or people per year",
    template="plotly_white",
    height=520,
)
figure.show()

print(f"active in 2000: {float(totals['active'][200]):.1f}")
print(f"prevalence in 2000: {float(prevalence[200]):.1f}")
